# Stage 03a — Aircraft Identity, Powertrain, Geography & Industry

Links every patent in a batch to the **real aircraft** it describes, and
characterises that aircraft, writing one workbook per batch:

    <data_matched>/<Batch_NN>/aircraft_identity_<Batch_NN>.xlsx

**Pipeline position:** runs off `batches.xlsx` + the PatSeer export, so it is
independent of the figure/label pipeline — you can run it before, during or
after `01a_wizard_feed`. It writes only its own file and touches no other
notebook's outputs.

## The four questions, and where each answer comes from

| Question | Columns | Signals, strongest first |
|---|---|---|
| Which aircraft? | `aircraft_name` | gazetteer → LLM → text mining |
| Is it electric? | `is_electric`, `powertrain` | gazetteer → LLM → keyword → SBERT |
| What are its numbers? | `pax`, `mtow_kg`, `payload_kg`, `cruise_speed_kmh`, `max_speed_kmh`, `range_km`, `endurance_min` | gazetteer → LLM → regex over patent text |
| Where / what for? | `assignee_country`, `region`, `pub_office`, `industry_primary` | assignee country code, publication office, SBERT zero-shot |

Every field carries its own `*_source` and `*_confidence`, and `needs_review`
flags the rows where a key field is missing or weak. That per-field provenance
is the point: **this stage is not expected to answer every question for every
patent**, and the sheet has to say which answers you can actually cite.

## What to expect, honestly

* **Most patents never name the aircraft.** Applicants write "an aircraft 100"
  precisely to avoid tying the claim to one product. Text mining will return
  nothing for the majority of the corpus — that is the correct result. The
  gazetteer and the LLM are the real name sources.
* **The shipped gazetteer has no spec numbers in it.** `reference/evtol_gazetteer.csv`
  ships with company → aircraft → powertrain filled and every numeric cell
  blank, with a `spec_source` column for you to fill from a citable source.
  A wrong MTOW in a thesis table is worse than a blank one.
* **A spec with `spec_source = regex`** was scraped out of the patent text and
  is usually an illustrative embodiment ("in one embodiment, a payload of about
  200 kg"), not the built aircraft. Treat those as review prompts.

## Running it

1. Set `BATCH_ID` in the next cell and run cells 1 → 7. That gives you the
   workbook with gazetteer + SBERT + regex answers.
2. For the LLM step, leave `LLM_MODE = "export"` (the default): the workbook's
   **LLM_Prompts** sheet gets one ready-made question per patent. Paste them
   into a chat, paste the replies into the `llm_answer` column, save, then run
   cell 8 to merge them in. Set `LLM_MODE = "api"` instead if you have an
   Anthropic API key and want it done automatically.
3. Correct anything wrong directly in the **Identity** sheet, and set that
   field's `*_source` cell to `human`. Re-running the notebook keeps your
   correction and everything in `notes` / `reviewed_by` / `llm_answer`.


In [ ]:
# ── Repo root on sys.path (same bootstrap as 01a_wizard_feed.ipynb) ────────
import sys
from pathlib import Path

_cwd = Path().resolve()
repo_root = None
for _candidate in [_cwd, *_cwd.parents]:
    if (_candidate / "src").exists() and (_candidate / "config.yaml").exists():
        repo_root = _candidate
        break
if repo_root is None:
    raise RuntimeError(f"Cannot find repo root from {_cwd}. Run from inside Patent-Labelling-Tools.")

for p in [str(repo_root), str(repo_root / "src")]:
    while p in sys.path:
        sys.path.remove(p)
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root))
print(f"repo_root : {repo_root}")

# ── Run configuration ──────────────────────────────────────────────────────
BATCH_ID = 1        # see data/batches.xlsx Summary sheet
LIMIT    = None     # e.g. 5 for a quick smoke test; None = whole batch

# LLM step:
#   "export" — write the prompts to the workbook, you paste answers back (no key needed)
#   "api"    — call the Anthropic API directly (needs ANTHROPIC_API_KEY or `ant auth login`)
#   "off"    — skip the LLM entirely
LLM_MODE  = "export"
LLM_MODEL = "claude-opus-5"

# SBERT is only needed for the industry classification, the powertrain fallback
# and candidate-name ranking. Set False for a fast, deterministic-only run.
USE_SBERT = True

# Only ask the LLM about patents the cheap signals could not name. Set False to
# ask about every patent (useful once, to measure how often the gazetteer and
# the LLM disagree).
LLM_ONLY_UNRESOLVED = True

In [ ]:
# ── Config, batch roster, PatSeer metadata, gazetteer ──────────────────────
import pandas as pd
from src.config_loader import load_config
from src.extractor import load_patseer_excel
from src import aircraft_identity as ai

cfg        = load_config()
sheet_name = f"Batch_{BATCH_ID:02d}"

batches_path = cfg["paths"]["batches_xlsx"]
if not batches_path.exists():
    raise FileNotFoundError(f"batches.xlsx not found at {batches_path} — run 00b1_grouping first.")

batch_df   = pd.read_excel(batches_path, sheet_name=sheet_name, dtype=str).fillna("")
patent_ids = batch_df["patent_id"].str.strip().tolist()
if LIMIT:
    patent_ids = patent_ids[:LIMIT]
    print(f"LIMIT={LIMIT} — processing only the first {len(patent_ids)} patent(s).")

# company_canonical / prototype_label are already on the batch sheet (written by
# 00b1_grouping via grouper.run_grouping), so the fuzzy assignee normalisation
# is NOT repeated here — reusing it keeps the company column in this workbook
# identical to the one every other stage sees.
batch_meta = {
    str(r["patent_id"]).strip(): {
        "company_canonical": (r.get("company_canonical") or "").strip() or None,
        "prototype_label":   (r.get("prototype_label") or "").strip() or None,
    }
    for _, r in batch_df.iterrows()
}

excel_index = load_patseer_excel(cfg["paths"]["patseer_excel"])
print(f"Batch {BATCH_ID}: {len(patent_ids)} patents; PatSeer index has {len(excel_index)} rows.")

gazetteer = ai.load_gazetteer(repo_root / "reference" / "evtol_gazetteer.csv")
_gaz_companies = {g["company_canonical"] for g in gazetteer}
print(f"Gazetteer: {len(gazetteer)} aircraft across {len(_gaz_companies)} companies.")

# How much of THIS batch the gazetteer can even reach. If this is near zero the
# gazetteer needs more rows before the LLM step is worth paying for.
_covered = sum(1 for p in patent_ids
               if (batch_meta.get(p, {}).get("company_canonical")) in _gaz_companies)
print(f"Gazetteer covers the company of {_covered}/{len(patent_ids)} patents in this batch.")

In [ ]:
# ── Classification text per patent ─────────────────────────────────────────
# SBERT (PatentSBERTa) truncates around 384 tokens, so the input is assembled
# signal-first: title and abstract state the invention, the first claim states
# it precisely, and the summary states why. The full Description is NOT
# available here by design — load_patseer_excel() deliberately skips it (see its
# docstring: most of it would be truncated away and what survives is boilerplate).
#
# The regex spec pass reads the SAME text, so any number stated only in the full
# description is out of reach. If you later decide those numbers matter, widen
# load_patseer_excel() rather than re-reading the Excel here.

def classify_text_for(pid: str) -> str:
    m = excel_index.get(pid, {})
    parts = [m.get("title"), m.get("abstract"), m.get("first_claim"),
             m.get("innovation_objective")]
    return "\n".join(p for p in parts if p)

def name_text_for(pid: str) -> str:
    # Name mining also reads the drawings description: a trade name, when it
    # appears at all, tends to appear in the figure captions.
    m = excel_index.get(pid, {})
    parts = [m.get("title"), m.get("abstract"), m.get("description_of_drawings"),
             m.get("innovation_objective")]
    return "\n".join(p for p in parts if p)

_missing = [p for p in patent_ids if p not in excel_index]
if _missing:
    print(f"⚠  {len(_missing)} patent(s) in the batch have no PatSeer row "
          f"(they get geography from the publication number only): {_missing[:5]}")

In [ ]:
# ── PatentSBERTa (optional) ────────────────────────────────────────────────
# Same model, same cache folder as 01a_wizard_feed / review_gpu_worker, so the
# weights are not downloaded a second time. HF_HUB_CACHE must be set BEFORE
# sentence_transformers imports huggingface_hub — it freezes the value at
# import time (same note 01a carries).
import os

sbert = None
if USE_SBERT:
    os.environ.setdefault("HF_HUB_CACHE", str(cfg["paths"]["sbert_cache"]))
    os.environ.setdefault("HF_HOME",      str(cfg["paths"]["sbert_cache"]))
    try:
        import torch
        from sentence_transformers import SentenceTransformer

        device = "cuda" if torch.cuda.is_available() else "cpu"
        sbert = SentenceTransformer(
            "AI-Growth-Lab/PatentSBERTa",
            cache_folder=str(cfg["paths"]["sbert_cache"]),
            device=device,
        )
        print(f"PatentSBERTa loaded on {device}.")
    except Exception as exc:
        # A missing GPU or an offline box degrades this stage to the
        # deterministic passes rather than failing the run.
        print(f"⚠  SBERT unavailable ({type(exc).__name__}: {exc}) — "
              f"running keyword/regex/gazetteer only.")
        sbert = None
else:
    print("USE_SBERT = False — running keyword/regex/gazetteer only.")

In [ ]:
# ── Pass A+B: gazetteer, powertrain, industry, name candidates, spec hints ─
from collections import Counter

signals = {}
for i, pid in enumerate(patent_ids, 1):
    meta  = excel_index.get(pid, {})
    ctext = classify_text_for(pid)
    ntext = name_text_for(pid)
    bmeta = batch_meta.get(pid, {})

    signals[pid] = {
        "meta":       meta,
        "batch_meta": bmeta,
        "gaz_hit":    ai.match_gazetteer(
            bmeta.get("company_canonical"), meta.get("app_year"), gazetteer,
            assignee_raw=meta.get("assignee"), text=ntext,
        ),
        # Keyword prior wins outright when it fires; SBERT is the fallback.
        "powertrain_pred": ai.classify_powertrain(ctext, sbert),
        "industry_pred":   ai.classify_industry(ctext, sbert),
        "name_candidates": ai.mine_name_candidates(ntext, pid, sbert),
        "spec_hints":      ai.extract_spec_hints(ctext),
    }
    if i % 50 == 0 or i == len(patent_ids):
        print(f"  {i}/{len(patent_ids)}")

# Coverage before the LLM runs — this is what the LLM step has to improve on.
_gaz_named = sum(1 for s in signals.values()
                 if (s["gaz_hit"] or {}).get("aircraft_name"))
_txt_named = sum(1 for s in signals.values() if s["name_candidates"])
_pt        = Counter(s["powertrain_pred"].get("value") for s in signals.values())
print(f"\nNamed by gazetteer     : {_gaz_named}/{len(patent_ids)}")
print(f"Name candidates in text: {_txt_named}/{len(patent_ids)}  "
      f"(low is expected — patents avoid naming the product)")
print(f"Powertrain             : {dict(_pt)}")

In [ ]:
# ── Pass C: the LLM ────────────────────────────────────────────────────────
# "export" (default) builds the prompts and stops — they land in the workbook's
# LLM_Prompts sheet for you to paste into a chat. "api" calls Claude directly.
# Either way the answers end up in the same {patent_id: dict} shape.

def _prompt_entry(pid):
    meta = excel_index.get(pid, {})
    return {
        "patent_id": pid,
        "company_canonical": batch_meta.get(pid, {}).get("company_canonical"),
        "assignee_raw": meta.get("assignee"),
        "app_year": meta.get("app_year"),
        "pub_year": meta.get("pub_year"),
        "title": meta.get("title"),
        "abstract": meta.get("abstract"),
    }

# Which patents to ask about. Asking only about the ones the gazetteer could not
# name is the cheap default; set LLM_ONLY_UNRESOLVED = False to ask about all of
# them and compare the LLM against the gazetteer on the rows both cover.
_ask = [pid for pid in patent_ids
        if not LLM_ONLY_UNRESOLVED
        or not (signals[pid]["gaz_hit"] or {}).get("aircraft_name")]

prompts_by_pid = {pid: ai.build_llm_prompt(_prompt_entry(pid)) for pid in _ask}
llm_answers = {}

if LLM_MODE == "api" and prompts_by_pid:
    print(f"Calling {LLM_MODEL} for {len(prompts_by_pid)} patent(s)...")
    llm_answers = ai.ask_claude(prompts_by_pid, model=LLM_MODEL)
    _errs = {p: a["_error"] for p, a in llm_answers.items() if a.get("_error")}
    _named = sum(1 for a in llm_answers.values() if a.get("aircraft_name"))
    print(f"LLM named {_named}/{len(llm_answers)}; {len(_errs)} error(s).")
    if _errs:
        print("  first errors:", list(_errs.items())[:3])
elif LLM_MODE == "export":
    print(f"LLM_MODE='export' — {len(prompts_by_pid)} prompt(s) will be written to the "
          f"LLM_Prompts sheet.\nPaste each reply into its `llm_answer` cell, save the "
          f"file, then run the ingest cell below.")
    print("\nSystem prompt to give the chat once, before the questions:\n")
    print(ai.LLM_SYSTEM_PROMPT)
else:
    print("LLM step skipped.")

In [ ]:
# ── Assemble rows and write aircraft_identity_<batch>.xlsx ─────────────────
rows, evidence = [], []
for pid in patent_ids:
    s = signals[pid]
    row, ev = ai.build_identity_row(
        patent_id=pid,
        batch=sheet_name,
        meta=s["meta"],
        batch_meta=s["batch_meta"],
        gaz_hit=s["gaz_hit"],
        powertrain_pred=s["powertrain_pred"],
        industry_pred=s["industry_pred"],
        name_candidates=s["name_candidates"],
        spec_hints=s["spec_hints"],
        llm_answer=llm_answers.get(pid),
    )
    rows.append(row)
    evidence.extend(ev)

prompt_rows = [
    {"patent_id": pid,
     "company_canonical": batch_meta.get(pid, {}).get("company_canonical"),
     "app_year": excel_index.get(pid, {}).get("app_year"),
     "llm_prompt": prompt,
     "llm_answer": None}
    for pid, prompt in prompts_by_pid.items()
]

# Same per-batch location the wizard feed writes to (data/matched/<batch>/), so
# every per-batch artefact for a batch sits in one folder.
data_matched = Path(cfg["paths"].get("data_matched", cfg["paths"]["data"]))
out_path = data_matched / sheet_name / f"aircraft_identity_{sheet_name}.xlsx"

# preserve_human=True: an existing file is backed up, then your `human`-sourced
# corrections, notes and pasted llm_answers are carried onto the new rows.
ai.export_identity_excel(rows, evidence, prompt_rows, out_path, preserve_human=True)
print(f"Wrote {len(rows)} patent row(s), {len(evidence)} evidence row(s) -> {out_path}")

In [ ]:
# ── Ingest LLM answers pasted into the workbook ────────────────────────────
# Run this AFTER filling the LLM_Prompts sheet's `llm_answer` column and saving.
# It re-reads the sheet, parses each reply (tolerant of ```json fences and
# surrounding prose), rebuilds every row with the LLM signal folded in at its
# proper precedence, and rewrites the workbook in place.
INGEST_LLM_ANSWERS = False   # flip to True once you have pasted the answers

if INGEST_LLM_ANSWERS:
    pasted = pd.read_excel(out_path, sheet_name="LLM_Prompts", dtype=object)
    llm_answers = {}
    _bad = []
    for _, r in pasted.iterrows():
        parsed = ai.parse_llm_answer(r.get("llm_answer"))
        if parsed:
            llm_answers[str(r["patent_id"]).strip()] = parsed
        elif pd.notna(r.get("llm_answer")) and str(r.get("llm_answer")).strip():
            _bad.append(str(r["patent_id"]).strip())

    print(f"Parsed {len(llm_answers)} answer(s).")
    if _bad:
        print(f"⚠  {len(_bad)} cell(s) had text that was not parseable JSON: {_bad[:5]}")

    rows, evidence = [], []
    for pid in patent_ids:
        s = signals[pid]
        row, ev = ai.build_identity_row(
            patent_id=pid, batch=sheet_name, meta=s["meta"], batch_meta=s["batch_meta"],
            gaz_hit=s["gaz_hit"], powertrain_pred=s["powertrain_pred"],
            industry_pred=s["industry_pred"], name_candidates=s["name_candidates"],
            spec_hints=s["spec_hints"], llm_answer=llm_answers.get(pid),
        )
        rows.append(row)
        evidence.extend(ev)

    ai.export_identity_excel(rows, evidence, prompt_rows, out_path, preserve_human=True)
    print(f"Rewrote {out_path} with the LLM answers merged in.")
else:
    print("INGEST_LLM_ANSWERS = False — nothing to ingest yet.")

In [ ]:
# ── Batch coverage report ──────────────────────────────────────────────────
# What this batch can actually support in the thesis. Read the "answered" counts
# as the population each downstream statistic is computed over — a 40 % naming
# rate is a real result about patent drafting practice, not a failed run.
ident = pd.DataFrame(rows, columns=ai.IDENTITY_COLUMNS)
n = len(ident)

def _pct(k):
    return f"{k:>4} / {n}  ({k / n:5.1%})" if n else "n/a"

print(f"=== {sheet_name} — {n} patents ===\n")
print("ANSWERED")
print(f"  aircraft_name    {_pct(ident['aircraft_name'].notna().sum())}")
print(f"  powertrain       {_pct(ident['powertrain'].notna().sum())}")
print(f"  industry         {_pct(ident['industry_primary'].notna().sum())}")
print(f"  any spec value   {_pct(ident[ai.SPEC_FIELDS].notna().any(axis=1).sum())}")
print(f"  needs_review     {_pct(ident['needs_review'].sum())}")

print("\nNAME SOURCE");      print(ident["aircraft_name_source"].value_counts(dropna=False).to_string())
print("\nIS ELECTRIC");      print(ident["is_electric"].value_counts(dropna=False).to_string())
print("\nPOWERTRAIN");       print(ident["powertrain"].value_counts(dropna=False).to_string())
print("\nREGION");           print(ident["region"].value_counts(dropna=False).to_string())
print("\nPUBLICATION OFFICE"); print(ident["pub_office"].value_counts(dropna=False).to_string())
print("\nINDUSTRY");         print(ident["industry_primary"].value_counts(dropna=False).to_string())
print("\nTOP COMPANIES");    print(ident["company_canonical"].value_counts().head(15).to_string())

# The rows to spend your review time on first: named, but weakly.
_weak = ident[(ident["aircraft_name"].notna())
              & (ident["aircraft_name_confidence"].fillna(0) < ai.NEEDS_REVIEW_BELOW)]
print(f"\nLow-confidence names to check first: {len(_weak)}")
if len(_weak):
    print(_weak[["patent_id", "company_canonical", "aircraft_name",
                 "aircraft_name_source", "aircraft_name_confidence"]].head(15).to_string(index=False))

In [ ]:
# ── Run every batch (optional) ─────────────────────────────────────────────
# Re-runs the whole notebook body for each Batch_NN sheet in batches.xlsx and
# writes one workbook per batch, then a combined view for the thesis tables.
# LLM_MODE is forced to "off" here — the export/paste loop is per batch and
# an unattended API sweep over the entire corpus should be a deliberate choice,
# not a side effect of running the last cell.
RUN_ALL_BATCHES = False

if RUN_ALL_BATCHES:
    import openpyxl

    wb_sheets = openpyxl.load_workbook(batches_path, read_only=True).sheetnames
    all_ident = []
    for sn in [s for s in wb_sheets if s.startswith("Batch_")]:
        bdf  = pd.read_excel(batches_path, sheet_name=sn, dtype=str).fillna("")
        pids = bdf["patent_id"].str.strip().tolist()
        bmet = {str(r["patent_id"]).strip(): {
                    "company_canonical": (r.get("company_canonical") or "").strip() or None,
                    "prototype_label":   (r.get("prototype_label") or "").strip() or None}
                for _, r in bdf.iterrows()}

        brows, bev, bprompts = [], [], []
        for pid in pids:
            meta  = excel_index.get(pid, {})
            ctext = "\n".join(x for x in [meta.get("title"), meta.get("abstract"),
                                          meta.get("first_claim"),
                                          meta.get("innovation_objective")] if x)
            ntext = "\n".join(x for x in [meta.get("title"), meta.get("abstract"),
                                          meta.get("description_of_drawings"),
                                          meta.get("innovation_objective")] if x)
            row, ev = ai.build_identity_row(
                patent_id=pid, batch=sn, meta=meta, batch_meta=bmet.get(pid, {}),
                gaz_hit=ai.match_gazetteer(bmet.get(pid, {}).get("company_canonical"),
                                           meta.get("app_year"), gazetteer,
                                           assignee_raw=meta.get("assignee"), text=ntext),
                powertrain_pred=ai.classify_powertrain(ctext, sbert),
                industry_pred=ai.classify_industry(ctext, sbert),
                name_candidates=ai.mine_name_candidates(ntext, pid, sbert),
                spec_hints=ai.extract_spec_hints(ctext),
            )
            brows.append(row)
            bev.extend(ev)
            bprompts.append({"patent_id": pid,
                             "company_canonical": bmet.get(pid, {}).get("company_canonical"),
                             "app_year": meta.get("app_year"),
                             "llm_prompt": ai.build_llm_prompt({**meta, "patent_id": pid,
                                 "assignee_raw": meta.get("assignee"),
                                 "company_canonical": bmet.get(pid, {}).get("company_canonical")}),
                             "llm_answer": None})

        p = data_matched / sn / f"aircraft_identity_{sn}.xlsx"
        ai.export_identity_excel(brows, bev, bprompts, p, preserve_human=True)
        print(f"{sn}: {len(brows)} rows -> {p}")
        all_ident.extend(brows)

    combined = Path(cfg["paths"]["data"]) / "Global Statistics" / "aircraft_identity_ALL.xlsx"
    combined.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(all_ident, columns=ai.IDENTITY_COLUMNS).to_excel(combined, index=False)
    print(f"\nCombined {len(all_ident)} rows -> {combined}")
else:
    print("RUN_ALL_BATCHES = False — single-batch run only.")

## Where this leaves you

The workbook is the deliverable — `Identity` joins onto anything keyed by
`patent_id`, and `Evidence` is the audit trail behind every value in it.

**To improve coverage, in order of payoff:**

1. **Grow `reference/evtol_gazetteer.csv`.** It is the only source with the
   precision to carry a spec number into the thesis. The batch report prints how
   many patents the gazetteer can reach by company — every company you add there
   converts a whole cluster of patents at once, and `company_canonical` must
   match a canonical name from `COMPANY_LOOKUP` in `src/grouper.py` or the row
   will never match.
2. **Fill the numeric columns** from type certificates, EASA/FAA publications
   or company datasheets, recording where each number came from in
   `spec_source`. Everything shipped is blank on purpose.
3. **Run the LLM step** for the companies the gazetteer does not cover, then
   spot-check its answers against the ones it *does* cover — that overlap is
   your measurement of how far to trust it, and it belongs in the methodology
   chapter.
4. **Correct the sheet by hand** where you know better, setting the field's
   `*_source` to `human`. Those survive every re-run.

**A caveat worth writing down in the thesis:** the naming rate is a property of
patent drafting practice, not of this method. Reporting "we could identify the
aircraft for N of M patents, and here is the source breakdown" is a stronger
result than a table that quietly implies all M were identified.
